# Vis-Head Causality Across Multi-Stage-Trained VLM Families

Compares head-level causal steering across training stages for four architecturally
distinct model families, per `mulltistage_trained_models.md`:

1. **Qwen-VL lineage** -- Qwen2-VL (base/instruct/agentic) + Qwen3-VL (instruct/thinking).
   Fully reuses this project's existing, validated pipeline.
2. **PaliGemma** -- `pt` (pretrain-only) vs. `mix` (fine-tuned) stages. Standard
   fixed-256-token architecture, new adapter (image_token_index, no chat template --
   PaliGemma uses task-prefix prompting).
3. **InternVL3.5** -- 4B and 8B, all 4 official stages (Pretrained/Instruct/MPO/final).
   Custom remote code, dynamic image tiling, `<IMG_CONTEXT>` token -- new adapter,
   more experimental than the above two.
4. **IDEFICS** -- base vs. instruct. **Architecturally incompatible** with this
   project's core steering mechanism: image information enters via a Perceiver
   Resampler (64 non-spatial latents) and gated cross-attention layers interleaved
   every 4th self-attention layer, not as tokens inside the text self-attention
   sequence. There is no per-region image-token KV position to bias the way Qwen/LLaVA/
   PaliGemma/InternVL allow. This section implements a **novel, experimental**
   adaptation instead: hooking the Perceiver Resampler's *internal* cross-attention
   (latents attending to raw, spatially-organized vision-encoder patches, before
   pooling) as the discovery/steering point. This is NOT the validated method used
   elsewhere in this project -- treat its results as exploratory.

**Discovery prompt**: exactly one phrasing throughout, `what_shows` -- `"What shows
the {name}?"` -- the single best-performing phrasing found in the 18-phrasing sweep
(`prompt_phrasing_vis_head_vs_causal.ipynb`).

**Scale**: 600 grids for Vis-Head discovery, 600 grids for causal MCQ evaluation, per
checkpoint. **Every parameter is set as a plain variable at the top of each section**
so you can edit and re-run any part independently. Each section has a **pilot cell**
(small N, fast, for catching bugs) before the **main cell** (full N).

**Metrics reported per checkpoint**: steered/baseline/location-cue accuracy, macro-F1,
macro one-vs-rest AUC (from letter-logit probabilities, single forward pass -- no
generation needed except where the checkpoint requires it), Vis-Head score mean/median/std,
and both raw and max-normalized (relative-to-own-max) Vis-Head scores.

In [ ]:
%matplotlib inline
import gc
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm

from vis_head.common import DEFAULT_SEED
from vis_head.gaze import aggregate_region_attention, collect_last_query_attentions, rank_heads_by_score
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import group_heads_by_layer, intervention_positions, make_static_attention_mask_hook, register_mask_hooks, remove_handles

DEVICE = "cuda:0"
SEED = DEFAULT_SEED
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_OPTIONS = 4
OPTION_LETTERS = ["A", "B", "C", "D"][:N_OPTIONS]
TOP_K_HEADS = 15

DISCOVERY_PROMPT = lambda name: f"What shows the {name}?"   # the single fixed discovery phrasing used throughout

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")


def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    n_distractors = min(N_OPTIONS - 1, len(other_names))
    distractor_idx = rng.choice(len(other_names), size=n_distractors, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter, "prompt": prompt}


def extract_letter_last(text, valid_letters):
    matches = re.findall(r"\b([" + "".join(valid_letters) + r"])\b", text.upper())
    return matches[-1] if matches else None


def macro_f1(y_true, y_pred):
    labels = sorted(set(y_true) | set(y_pred))
    return float(f1_score(y_true, y_pred, labels=labels, average="macro"))


def macro_auc(y_true, probs, classes):
    y_true_bin = label_binarize(y_true, classes=classes)
    try:
        return float(roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr"))
    except ValueError:
        return float("nan")


def score_stats(scores):
    flat = scores.reshape(-1)
    return {"mean": float(flat.mean()), "median": float(np.median(flat)), "std": float(flat.std()), "max": float(flat.max())}


def normalized_scores(scores):
    m = scores.max()
    return scores / m if m > 0 else scores.copy()


def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


def get_letter_token_ids(tokenizer, letters):
    """Map each option letter to the single-token ids that could represent it
    as a generated token (bare and leading-space variants)."""
    ids = {}
    for letter in letters:
        candidates = set()
        for cand in (letter, f" {letter}"):
            enc = tokenizer.encode(cand, add_special_tokens=False)
            if len(enc) == 1:
                candidates.add(enc[0])
        if not candidates:
            raise ValueError(f"No single-token encoding found for letter {letter!r}")
        ids[letter] = sorted(candidates)
    return ids


def run_mcq_generate(model, inputs, prompt_length, heads_by_layer, register_hooks_fn,
                      max_new_tokens, letter_token_ids, all_letter_ids_flat, id_to_letter):
    """Uniform MCQ evaluation for every family/checkpoint in this notebook,
    including CoT/reasoning models that need many tokens before answering:
    generate with output_scores, scan the ACTUAL generated token ids for the
    LAST occurrence of any option-letter token (works whether the model answers
    immediately or after a long reasoning trace), and read real logits at that
    exact decode step for a genuine probability vector (accuracy/F1/AUC)."""
    handles = register_hooks_fn(heads_by_layer) if heads_by_layer is not None else []
    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  output_scores=True, return_dict_in_generate=True)
    finally:
        remove_handles(handles)

    gen_ids = out.sequences[0, prompt_length:].tolist()
    answer_step = None
    for step in range(len(gen_ids) - 1, -1, -1):
        if gen_ids[step] in all_letter_ids_flat:
            answer_step = step
            break
    if answer_step is None:
        return {"predicted": None, "probs": np.zeros(N_OPTIONS), "correct": False}

    predicted = id_to_letter[gen_ids[answer_step]]
    step_logits = out.scores[answer_step][0].float()
    letter_logits = [step_logits[letter_token_ids[l]].max().item() for l in OPTION_LETTERS]
    probs = torch.softmax(torch.tensor(letter_logits), dim=0).numpy()
    return {"predicted": predicted, "probs": probs, "correct": None}   # "correct" filled in by caller (needs ground truth)


def fit_and_eval_probe(hidden_states, labels, seed=SEED):
    X = np.stack(hidden_states)
    y = np.array(labels)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = macro_f1(list(y_test), list(y_pred))
    y_proba = clf.predict_proba(X_test)
    auc = macro_auc(list(y_test), y_proba, list(clf.classes_))
    return {"accuracy": acc, "f1": f1, "auc": auc}


all_results = {}   # (family, checkpoint_tag) -> summary dict, filled in by every section below

---

# Section 1: Qwen-VL lineage

5 checkpoints: Qwen2-VL {base, instruct, agentic=UI-TARS-7B-DPO} + Qwen3-VL
{instruct, thinking}. Fully reuses this project's existing, validated pipeline.

Discovery method: standard final-query attention for all checkpoints **except**
`qwen3_thinking`, which uses the CoT-trace method (`collect_cot_trace_region_attention`)
-- the standard method was empirically found to select heads with no causal effect
for this checkpoint (see `cot_vis_head_discovery_qwen3_thinking.ipynb`).

Every parameter below is a plain variable -- edit and re-run freely.

In [ ]:
# ----------------------------- Section 1 configuration -----------------------------
QWEN_CHECKPOINTS = {
    "qwen2_base":     {"model_id": "Qwen/Qwen2-VL-7B",              "max_new_tokens": 6,   "cot_discovery": False, "needs_template_workaround": True},
    "qwen2_instruct": {"model_id": "Qwen/Qwen2-VL-7B-Instruct",     "max_new_tokens": 6,   "cot_discovery": False, "needs_template_workaround": False},
    "qwen2_agentic":  {"model_id": "ByteDance-Seed/UI-TARS-7B-DPO", "max_new_tokens": 6,   "cot_discovery": False, "needs_template_workaround": False},
    "qwen3_instruct": {"model_id": "Qwen/Qwen3-VL-8B-Instruct",     "max_new_tokens": 6,   "cot_discovery": False, "needs_template_workaround": False},
    "qwen3_thinking": {"model_id": "Qwen/Qwen3-VL-8B-Thinking",     "max_new_tokens": 600, "cot_discovery": True,  "needs_template_workaround": False},
}
QWEN_N_DISCOVERY = 600     # edit for a smaller/larger run
QWEN_N_CAUSAL = 600
QWEN_COT_DISCOVERY_MAX_NEW_TOKENS = 150   # only used when cot_discovery=True

from vis_head.gaze import collect_cot_trace_region_attention
from transformers import AutoProcessor as _AutoProcessor

_qwen_template_processor = _AutoProcessor.from_pretrained(QWEN_CHECKPOINTS["qwen2_instruct"]["model_id"])


def qwen_render_chat_text(prompt):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    rendered = _qwen_template_processor.apply_chat_template([messages], tokenize=False, add_generation_prompt=True)
    return rendered[0] if isinstance(rendered, list) else rendered


def qwen_prepare_inputs(processor, image, prompt, needs_workaround):
    if needs_workaround:
        text = qwen_render_chat_text(prompt)
        return processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)
    return prepare_inputs(processor, image, prompt, DEVICE)


def run_qwen_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    model, processor = load_model_and_processor(model_id=cfg["model_id"], device=DEVICE)
    n_layers, n_heads, spatial_merge = model_dims(model)
    print(f"{n_layers} layers x {n_heads} heads")
    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = qwen_prepare_inputs(processor, grid.grid, prompt, cfg["needs_template_workaround"])
            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            if cfg["cot_discovery"]:
                img_start, img_end = find_image_token_range(inputs, processor)
                positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
                scores = collect_cot_trace_region_attention(
                    model=model, inputs=inputs, target_positions=positions[target_cell],
                    img_start=img_start, img_end=img_end, n_layers=n_layers, n_heads=n_heads,
                    max_new_tokens=QWEN_COT_DISCOVERY_MAX_NEW_TOKENS)
                raw_sum += scores
            else:
                attn = collect_last_query_attentions(model, inputs)
                region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
                raw_sum += region_attn[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")
    vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    ranked = rank_heads_by_score(vis_head_scores)
    heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]])

    def register_hooks(heads_by_layer_arg):
        return []   # placeholder, overwritten below for the actual steered call

    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, steer):
        inputs = qwen_prepare_inputs(processor, sample["grid"].grid, sample["prompt"], cfg["needs_template_workaround"])
        prompt_length = int(inputs["input_ids"].shape[1])

        def hooks_fn(_unused):
            img_start, img_end = find_image_token_range(inputs, processor)
            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in heads_by_layer.items()
            }
            return register_mask_hooks(model, hook_by_layer)

        result = run_mcq_generate(model, inputs, prompt_length, heads_by_layer if steer else None,
                                   hooks_fn, cfg["max_new_tokens"], letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, steered_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ", leave=False):
        baseline_res.append(run_one(sample, steer=False))
        steered_res.append(run_one(sample, steer=True))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    s_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in steered_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    s_probs = np.stack([r["probs"] for r in steered_res])

    stats = score_stats(vis_head_scores)
    norm_stats = score_stats(normalized_scores(vis_head_scores))
    summary = {
        "family": "qwen", "checkpoint": tag, "model_id": cfg["model_id"],
        "vh_mean": stats["mean"], "vh_median": stats["median"], "vh_std": stats["std"], "vh_max": stats["max"],
        "vh_norm_mean": norm_stats["mean"], "vh_norm_median": norm_stats["median"], "vh_norm_std": norm_stats["std"],
        "baseline_acc": np.mean([r["correct"] for r in baseline_res]),
        "steered_acc": np.mean([r["correct"] for r in steered_res]),
        "baseline_f1": macro_f1(y_true, b_pred), "steered_f1": macro_f1(y_true, s_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "steered_auc": macro_auc(y_true, s_probs, OPTION_LETTERS),
    }
    b_acc_v, s_acc_v, s_auc_v = summary["baseline_acc"], summary["steered_acc"], summary["steered_auc"]
    print(f"  [{tag}] baseline_acc={b_acc_v:.3f}  steered_acc={s_acc_v:.3f}  steered_auc={s_auc_v:.3f}")

    free_gpu(model, processor)
    return summary

## Section 1 -- Pilot run (small N, catches bugs fast)

In [ ]:
PILOT_N_DISCOVERY = 20
PILOT_N_CAUSAL = 20

for tag, cfg in QWEN_CHECKPOINTS.items():
    result = run_qwen_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("qwen", tag + "_PILOT")] = result

pd.DataFrame([v for k, v in all_results.items() if k[0] == "qwen" and "PILOT" in k[1]])

## Section 1 -- Main run (N=QWEN_N_DISCOVERY discovery, QWEN_N_CAUSAL causal, default 600/600)

In [ ]:
for tag, cfg in QWEN_CHECKPOINTS.items():
    result = run_qwen_checkpoint(tag, cfg, QWEN_N_DISCOVERY, QWEN_N_CAUSAL)
    all_results[("qwen", tag)] = result

qwen_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "qwen" and "PILOT" not in k[1]])
qwen_df.to_csv("logs/multistage_qwen_results.csv", index=False)
pd.set_option("display.width", 200)
print(qwen_df.to_string(index=False))

---

# Section 2: PaliGemma

`pt` (pretrain-only, no instruction tuning) vs. `mix` (instruction/task fine-tuned)
-- the closest PaliGemma has to a base/instruct pair. Standard, non-dynamic
architecture: fixed 256 image tokens (16x16 SigLIP patches, `image_size=224`,
`patch_size=14`), `image_token_index=257152` (single int, like LLaVA-1.5/Gemma-3).
Confirmed `model.model.language_model.layers` module path (matches an existing
`language_model_layers()` candidate path -- no shared-library change needed).

**Important difference from every other family in this notebook: PaliGemma has NO
chat template.** It uses fixed task-prefix prompting (e.g. `"answer en {question}"`)
rather than a conversational format -- confirmed via `AutoProcessor` (`chat_template`
is `None`). Prompts below are adapted to this convention rather than reusing the
chat-style MCQ prompt used elsewhere.

In [ ]:
# ----------------------------- Section 2 configuration -----------------------------
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration

PALIGEMMA_CHECKPOINTS = {
    "paligemma_pt":  {"model_id": "google/paligemma-3b-pt-224"},
    "paligemma_mix": {"model_id": "google/paligemma-3b-mix-224"},
}
PALIGEMMA_N_DISCOVERY = 600
PALIGEMMA_N_CAUSAL = 600
PALIGEMMA_IMAGE_TOKEN_ID = 257152
PALIGEMMA_MAX_NEW_TOKENS = 6

import vis_head.gaze as _gaze_module


def paligemma_find_image_token_range(inputs, processor=None):
    ids = inputs["input_ids"][0].tolist()
    positions = [i for i, t in enumerate(ids) if t == PALIGEMMA_IMAGE_TOKEN_ID]
    if not positions:
        raise ValueError("No PaliGemma image tokens found.")
    return positions[0], positions[-1] + 1


def paligemma_prepare_inputs(processor, image, prompt):
    # PaliGemma convention: no chat template, "answer en {question}" task prefix.
    text = f"answer en {prompt}"
    return processor(text=text, images=image, return_tensors="pt").to(DEVICE)


def paligemma_assign_grid(rows, cols, grid_side=16):
    fake_thw = torch.tensor([[1, grid_side, grid_side]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=rows, cols=cols, spatial_merge=1)


def run_paligemma_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    processor = AutoProcessor.from_pretrained(cfg["model_id"])
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, attn_implementation="eager"
    ).to(DEVICE)
    model.eval()
    n_layers = model.config.text_config.num_hidden_layers
    n_heads = model.config.text_config.num_attention_heads
    grid_side = model.config.vision_config.image_size // model.config.vision_config.patch_size
    print(f"{n_layers} layers x {n_heads} heads, vision patch grid {grid_side}x{grid_side}")
    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    _gaze_module.find_image_token_range = paligemma_find_image_token_range

    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = paligemma_prepare_inputs(processor, grid.grid, prompt)
            region_ids, _ = paligemma_assign_grid(ROWS, COLS, grid_side)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            raw_sum += region_attn[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")
    vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    ranked = rank_heads_by_score(vis_head_scores)
    heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]])

    def build_paligemma_mcq_sample(rng):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        correct_name = grid.cell_names[target_cell]
        other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
        distractor_idx = rng.choice(len(other_names), size=min(N_OPTIONS - 1, len(other_names)), replace=False)
        distractors = [other_names[i] for i in distractor_idx]
        options = [correct_name] + distractors
        order = rng.permutation(len(options))
        options = [options[i] for i in order]
        correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
        option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
        prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
        return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter, "prompt": prompt}

    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_paligemma_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, steer):
        inputs = paligemma_prepare_inputs(processor, sample["grid"].grid, sample["prompt"])
        prompt_length = int(inputs["input_ids"].shape[1])

        def hooks_fn(_unused):
            img_start, img_end = paligemma_find_image_token_range(inputs)
            region_ids, _ = paligemma_assign_grid(ROWS, COLS, grid_side)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in heads_by_layer.items()
            }
            return register_mask_hooks(model, hook_by_layer)

        result = run_mcq_generate(model, inputs, prompt_length, heads_by_layer if steer else None,
                                   hooks_fn, PALIGEMMA_MAX_NEW_TOKENS, letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, steered_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ", leave=False):
        baseline_res.append(run_one(sample, steer=False))
        steered_res.append(run_one(sample, steer=True))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    s_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in steered_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    s_probs = np.stack([r["probs"] for r in steered_res])

    stats = score_stats(vis_head_scores)
    norm_stats = score_stats(normalized_scores(vis_head_scores))
    summary = {
        "family": "paligemma", "checkpoint": tag, "model_id": cfg["model_id"],
        "vh_mean": stats["mean"], "vh_median": stats["median"], "vh_std": stats["std"], "vh_max": stats["max"],
        "vh_norm_mean": norm_stats["mean"], "vh_norm_median": norm_stats["median"], "vh_norm_std": norm_stats["std"],
        "baseline_acc": np.mean([r["correct"] for r in baseline_res]),
        "steered_acc": np.mean([r["correct"] for r in steered_res]),
        "baseline_f1": macro_f1(y_true, b_pred), "steered_f1": macro_f1(y_true, s_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "steered_auc": macro_auc(y_true, s_probs, OPTION_LETTERS),
    }
    b_acc_v, s_acc_v, s_auc_v = summary["baseline_acc"], summary["steered_acc"], summary["steered_auc"]
    print(f"  [{tag}] baseline_acc={b_acc_v:.3f}  steered_acc={s_acc_v:.3f}  steered_auc={s_auc_v:.3f}")

    free_gpu(model, processor)
    return summary

## Section 2 -- Pilot run

In [ ]:
for tag, cfg in PALIGEMMA_CHECKPOINTS.items():
    result = run_paligemma_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("paligemma", tag + "_PILOT")] = result

pd.DataFrame([v for k, v in all_results.items() if k[0] == "paligemma" and "PILOT" in k[1]])

## Section 2 -- Main run

In [ ]:
for tag, cfg in PALIGEMMA_CHECKPOINTS.items():
    result = run_paligemma_checkpoint(tag, cfg, PALIGEMMA_N_DISCOVERY, PALIGEMMA_N_CAUSAL)
    all_results[("paligemma", tag)] = result

paligemma_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "paligemma" and "PILOT" not in k[1]])
paligemma_df.to_csv("logs/multistage_paligemma_results.csv", index=False)
print(paligemma_df.to_string(index=False))

---

# Section 3: InternVL3.5 (4B and 8B, all 4 official stages)

`Pretrained` (CPT) -> `Instruct` (CPT+SFT) -> `MPO` (CPT+SFT+MPO) -> final (CPT+SFT+MPO+CascadeRL),
for both 4B and 8B (8 checkpoints total). All four stages per size confirmed
**architecturally identical** via `AutoConfig` (`mulltistage_trained_models.md`
verification pass): 8B is 36 layers x 32 heads, backbone is **Qwen3** paired with
an **InternViT-6B** vision encoder.

**Custom remote code** (`trust_remote_code=True`) -- no standard `AutoProcessor`
image handling (`AutoProcessor` resolves to a bare text tokenizer for this model).
Image preprocessing (dynamic tiling into up to 12 448x448 patches + a thumbnail,
per InternVL's documented convention) and the `<IMG_CONTEXT>` placeholder-token
insertion are implemented manually below, following InternVL's standard usage
pattern. **This section is more experimental than Sections 1-2** -- less
incrementally validated than the LLaVA/Qwen adapters elsewhere in this project.
Trust-remote-code is a real supply-chain consideration: only run this against
checkpoints you trust.

In [ ]:
# ----------------------------- Section 3 configuration -----------------------------
from transformers import AutoModel, AutoTokenizer
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

INTERNVL_CHECKPOINTS = {
    "internvl35_4b_pretrained": {"model_id": "OpenGVLab/InternVL3_5-4B-Pretrained"},
    "internvl35_4b_instruct":   {"model_id": "OpenGVLab/InternVL3_5-4B-Instruct"},
    "internvl35_4b_mpo":        {"model_id": "OpenGVLab/InternVL3_5-4B-MPO"},
    "internvl35_4b_final":      {"model_id": "OpenGVLab/InternVL3_5-4B"},
    "internvl35_8b_pretrained": {"model_id": "OpenGVLab/InternVL3_5-8B-Pretrained"},
    "internvl35_8b_instruct":   {"model_id": "OpenGVLab/InternVL3_5-8B-Instruct"},
    "internvl35_8b_mpo":        {"model_id": "OpenGVLab/InternVL3_5-8B-MPO"},
    "internvl35_8b_final":      {"model_id": "OpenGVLab/InternVL3_5-8B"},
}
INTERNVL_N_DISCOVERY = 600
INTERNVL_N_CAUSAL = 600
INTERNVL_MAX_NEW_TOKENS = 6
INTERNVL_IMAGE_SIZE = 448
INTERNVL_PATCH_SIZE = 14
INTERNVL_DOWNSAMPLE = 0.5   # from vision_config.downsample_ratio -- token count per tile = (448/14 * 0.5)^2 = 256
INTERNVL_TOKENS_PER_TILE = int((INTERNVL_IMAGE_SIZE // INTERNVL_PATCH_SIZE * INTERNVL_DOWNSAMPLE) ** 2)   # 256
INTERNVL_IMAGENET_MEAN = (0.485, 0.456, 0.406)
INTERNVL_IMAGENET_STD = (0.229, 0.224, 0.225)

import vis_head.gaze as _gaze_module

_internvl_transform = T.Compose([
    T.Resize((INTERNVL_IMAGE_SIZE, INTERNVL_IMAGE_SIZE), interpolation=InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=INTERNVL_IMAGENET_MEAN, std=INTERNVL_IMAGENET_STD),
])


def internvl_prepare_inputs(model, tokenizer, image, prompt, device):
    """Single-tile-only preprocessing (no dynamic multi-tile splitting, since our
    grid images are already square and modest resolution) -- one IMG_CONTEXT block
    of INTERNVL_TOKENS_PER_TILE tokens, following InternVL's documented convention
    (see modeling_internvl_chat.py's `chat()` method for the reference pattern this
    reimplements manually so we can hook attention/steering directly)."""
    pixel_values = _internvl_transform(image.convert("RGB")).unsqueeze(0).to(device=device, dtype=model.dtype)
    img_context_token_id = tokenizer.convert_tokens_to_ids("<IMG_CONTEXT>")
    image_tokens = "<IMG_CONTEXT>" * INTERNVL_TOKENS_PER_TILE
    full_prompt = f"<img>{image_tokens}</img>\n{prompt}"
    # Exact "internvl2_5" conversation-template format used by InternVL's own
    # .chat() method (confirmed via direct source inspection) -- the system-message
    # block is required; omitting it (as an earlier version of this adapter did)
    # produced empty generations on every sample, since the model was never trained
    # on inputs lacking it.
    system_message = getattr(model, "system_message", "")
    chat_text = f"<|im_start|>system\n{system_message}<|im_end|>\n<|im_start|>user\n{full_prompt}<|im_end|>\n<|im_start|>assistant\n"
    model_inputs = tokenizer(chat_text, return_tensors="pt").to(device)
    return {"input_ids": model_inputs["input_ids"], "attention_mask": model_inputs["attention_mask"],
            "pixel_values": pixel_values, "img_context_token_id": img_context_token_id}


def internvl_find_image_token_range(inputs, processor=None):
    ids = inputs["input_ids"][0].tolist()
    ctx_id = inputs["img_context_token_id"]
    positions = [i for i, t in enumerate(ids) if t == ctx_id]
    if not positions:
        raise ValueError("No InternVL IMG_CONTEXT tokens found.")
    return positions[0], positions[-1] + 1


def internvl_assign_grid(rows, cols):
    grid_side = int(INTERNVL_TOKENS_PER_TILE ** 0.5)   # 16
    fake_thw = torch.tensor([[1, grid_side, grid_side]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=rows, cols=cols, spatial_merge=1)


class _InternVLForwardAdapter:
    """model.forward(pixel_values=..., input_ids=..., ...) requires image_flags to
    mark which rows of pixel_values are real images (InternVL supports batched
    multi-tile input) -- always 1 tile here, so image_flags is a constant ones
    vector. Wraps the custom-code model call so the rest of the pipeline
    (collect_last_query_attentions, run_generation-equivalent) sees a plain
    model(**inputs) / model.generate(**inputs) interface like every other family.
    """
    def __init__(self, model):
        self.model = model

    def __call__(self, **kwargs):
        kwargs = dict(kwargs)
        img_context_token_id = kwargs.pop("img_context_token_id", None)
        if img_context_token_id is not None:
            self.model.img_context_token_id = img_context_token_id   # InternVL's custom forward() reads this as a model attribute, not an argument
        kwargs["image_flags"] = torch.ones(kwargs["pixel_values"].shape[0], dtype=torch.long, device=kwargs["pixel_values"].device)
        return self.model(**kwargs)

    def generate(self, **kwargs):
        kwargs = dict(kwargs)
        img_context_token_id = kwargs.pop("img_context_token_id")
        self.model.img_context_token_id = img_context_token_id
        pixel_values = kwargs.pop("pixel_values")
        input_ids = kwargs.pop("input_ids")
        attention_mask = kwargs.pop("attention_mask")
        return self.model.generate(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, **kwargs)


def run_internvl_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    tokenizer = AutoTokenizer.from_pretrained(cfg["model_id"], trust_remote_code=True, use_fast=False)
    raw_model = AutoModel.from_pretrained(
        cfg["model_id"], torch_dtype=torch.bfloat16, trust_remote_code=True, attn_implementation="eager"
    ).to(DEVICE)
    raw_model.eval()
    model = _InternVLForwardAdapter(raw_model)
    n_layers = raw_model.config.llm_config.num_hidden_layers
    n_heads = raw_model.config.llm_config.num_attention_heads
    print(f"{n_layers} layers x {n_heads} heads")
    letter_token_ids = get_letter_token_ids(tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    _gaze_module.find_image_token_range = internvl_find_image_token_range

    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = internvl_prepare_inputs(raw_model, tokenizer, grid.grid, prompt, DEVICE)
            region_ids, _ = internvl_assign_grid(ROWS, COLS)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=None, region_ids=region_ids, n_regions=N_CELLS)
            raw_sum += region_attn[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")
    vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    ranked = rank_heads_by_score(vis_head_scores)
    heads_by_layer = group_heads_by_layer([(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]])

    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, steer):
        inputs = internvl_prepare_inputs(raw_model, tokenizer, sample["grid"].grid, sample["prompt"], DEVICE)
        prompt_length = int(inputs["input_ids"].shape[1])

        def hooks_fn(_unused):
            img_start, img_end = internvl_find_image_token_range(inputs)
            region_ids, _ = internvl_assign_grid(ROWS, COLS)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in heads_by_layer.items()
            }
            return register_mask_hooks(raw_model, hook_by_layer)

        # InternVL's custom generate() returns ONLY the newly generated tokens (it
        # internally converts input_ids+image to inputs_embeds, and HF's generate()
        # does not prepend inputs_embeds-derived positions back into `sequences` --
        # confirmed via direct inspection: sequences.shape == (1, max_new_tokens),
        # not (1, prompt_len + max_new_tokens) like every other family in this
        # notebook). Slicing with the real prompt_length here would cut past the
        # end of the tensor and always return an empty result, so pass 0 instead.
        result = run_mcq_generate(model, inputs, 0, heads_by_layer if steer else None,
                                   hooks_fn, INTERNVL_MAX_NEW_TOKENS, letter_token_ids, all_letter_ids_flat, id_to_letter)
        result["correct"] = result["predicted"] == sample["correct_letter"]
        return result

    baseline_res, steered_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ", leave=False):
        baseline_res.append(run_one(sample, steer=False))
        steered_res.append(run_one(sample, steer=True))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    s_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in steered_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    s_probs = np.stack([r["probs"] for r in steered_res])

    stats = score_stats(vis_head_scores)
    norm_stats = score_stats(normalized_scores(vis_head_scores))
    summary = {
        "family": "internvl35", "checkpoint": tag, "model_id": cfg["model_id"],
        "vh_mean": stats["mean"], "vh_median": stats["median"], "vh_std": stats["std"], "vh_max": stats["max"],
        "vh_norm_mean": norm_stats["mean"], "vh_norm_median": norm_stats["median"], "vh_norm_std": norm_stats["std"],
        "baseline_acc": np.mean([r["correct"] for r in baseline_res]),
        "steered_acc": np.mean([r["correct"] for r in steered_res]),
        "baseline_f1": macro_f1(y_true, b_pred), "steered_f1": macro_f1(y_true, s_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "steered_auc": macro_auc(y_true, s_probs, OPTION_LETTERS),
    }
    b_acc_v, s_acc_v, s_auc_v = summary["baseline_acc"], summary["steered_acc"], summary["steered_auc"]
    print(f"  [{tag}] baseline_acc={b_acc_v:.3f}  steered_acc={s_acc_v:.3f}  steered_auc={s_auc_v:.3f}")

    free_gpu(raw_model, model, tokenizer)
    return summary

## Section 3 -- Pilot run

In [ ]:
for tag, cfg in INTERNVL_CHECKPOINTS.items():
    result = run_internvl_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("internvl35", tag + "_PILOT")] = result

pd.DataFrame([v for k, v in all_results.items() if k[0] == "internvl35" and "PILOT" in k[1]])

## Section 3 -- Main run

In [ ]:
for tag, cfg in INTERNVL_CHECKPOINTS.items():
    result = run_internvl_checkpoint(tag, cfg, INTERNVL_N_DISCOVERY, INTERNVL_N_CAUSAL)
    all_results[("internvl35", tag)] = result

internvl_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "internvl35" and "PILOT" not in k[1]])
internvl_df.to_csv("logs/multistage_internvl_results.csv", index=False)
print(internvl_df.to_string(index=False))

---

# Section 4: IDEFICS (base vs. instruct) -- EXPERIMENTAL, non-standard method

**Architecturally incompatible with this project's core steering mechanism.**
IDEFICS injects image information via a **Perceiver Resampler** (64 fixed,
non-spatial latent vectors, `IdeficsPerceiverResampler`) followed by **gated
cross-attention layers** (`IdeficsGatedCrossAttentionLayer`) interleaved every 4th
self-attention layer -- confirmed via live module inspection. The LM's
`self_attn` modules never see image tokens directly (`IdeficsAttention.q/k/v_proj`
all take `hidden_size=4096`, purely textual), so this project's `attention_mask`-hook
steering method (biasing specific image-token KV positions inside `self_attn`) has
**no target to act on** for IDEFICS -- there is no per-region image-token KV
position anywhere in the LM's self-attention.

The only place spatially-organized image information exists at all is *inside* the
Perceiver Resampler itself, where 64 latent queries cross-attend to the raw,
spatially-organized vision-encoder patch tokens (`IdeficsPerceiverAttention.forward`,
confirmed via source inspection -- computes `scores = einsum(q, k)` directly, with
**no `attention_mask` kwarg to hook** the way every other family in this notebook
allows). This section implements a genuinely different, novel intervention:

- **Discovery**: capture the Resampler's internal attention (context->latents) via
  a forward hook wrapping `IdeficsPerceiverAttention.forward`, and measure how much
  of it lands on the target grid region's raw vision patches, per resampler
  block/head (analogous to a "Vis-Head" score, but for resampler heads, not LM heads).
- **Steering**: monkey-patch `IdeficsPerceiverAttention.forward` to add a static
  additive bias to `scores` before the softmax (the same `boost_suppress` recipe
  used elsewhere, reimplemented at this different attention site since there is no
  `attention_mask` kwarg to intercept via a clean hook).

**This is exploratory and has not been validated the way the other three sections'
methods have been (no prior pilot precedent for this specific mechanism, unlike the
CoT-trace method which was validated before being trusted). Treat results here as a
first look, not a confirmed finding.**

In [ ]:
# ----------------------------- Section 4 configuration -----------------------------
from transformers import IdeficsForVisionText2Text, AutoProcessor as _AutoProcessor2
from transformers.models.idefics import perceiver as _idefics_perceiver

IDEFICS_CHECKPOINTS = {
    "idefics_base":     {"model_id": "HuggingFaceM4/idefics-9b"},
    "idefics_instruct": {"model_id": "HuggingFaceM4/idefics-9b-instruct"},
}
IDEFICS_N_DISCOVERY = 600
IDEFICS_N_CAUSAL = 600
IDEFICS_MAX_NEW_TOKENS = 12   # IDEFICS tends to answer with a fuller phrase ("Answer: D) barbershop.") before <end_of_utterance>, confirmed via diagnostic
IDEFICS_SWAP_BIAS = 10000.0   # same magnitude as the project default elsewhere
IDEFICS_N_RESAMPLER_BLOCKS = 6   # perceiver_config.resampler_depth
IDEFICS_N_RESAMPLER_HEADS = 16   # perceiver_config.resampler_n_heads
IDEFICS_N_LATENTS = 64            # perceiver_config.resampler_n_latents
IDEFICS_VISION_GRID_SIDE = 16      # 224 / 14 = 16x16 raw vision patches (pre-resampling)


def idefics_prepare_inputs(processor, image, prompt, device):
    # IDEFICS role-marker convention -- confirmed necessary via diagnostic: plain
    # `text=prompt` (no markers) produces an immediate empty generation (the model
    # emits <end_of_utterance> as its very first token with no role cue to respond to).
    text = f"User: <image>{prompt}<end_of_utterance>\nAssistant:"
    inputs = processor(images=[image], text=text, return_tensors="pt")
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}


def idefics_assign_grid(rows, cols):
    fake_thw = torch.tensor([[1, IDEFICS_VISION_GRID_SIDE, IDEFICS_VISION_GRID_SIDE]])
    return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=rows, cols=cols, spatial_merge=1)


class _ResamplerAttentionRecorder:
    """Wraps IdeficsPerceiverAttention.forward to record post-softmax attention
    (context positions x heads) without changing behavior -- used for discovery."""
    def __init__(self):
        self.records = []   # list of (block_idx, attn: np.ndarray[n_heads, n_context])
        self._orig_forwards = {}

    def attach(self, resampler):
        for block_idx, block in enumerate(resampler.blocks):
            attn_module = block[0]   # IdeficsPerceiverAttention
            orig_forward = attn_module.forward
            self._orig_forwards[block_idx] = (attn_module, orig_forward)

            def make_patched(mod, orig_fwd, b_idx):
                def patched(context, latents):
                    context_n = mod.context_layer_norm(context)
                    latents_n = mod.latents_layer_norm(latents)
                    batch_size, seq_length, embed_dim = context_n.shape[:3]
                    q = mod.q_proj(latents_n)
                    k = mod.k_proj(torch.cat([context_n, latents_n], dim=-2))
                    v = mod.v_proj(torch.cat([context_n, latents_n], dim=-2))
                    q, k, v = [x.reshape(batch_size, x.shape[1], mod.n_heads, mod.head_dim).transpose(1, 2) for x in (q, k, v)]
                    if mod.qk_layer_norms:
                        q = mod.q_layer_norm(q)
                        k = mod.k_layer_norm(k)
                    scores = torch.einsum("... i d, ... j d -> ... i j", q * mod.qk_scale, k)
                    stabilized = scores - scores.amax(dim=-1, keepdim=True).detach()
                    attn = stabilized.softmax(dim=-1)
                    self.records.append((b_idx, attn[0, :, :, :seq_length].detach().float().cpu().numpy()))
                    resampled = torch.einsum("... i j, ... j d -> ... i d", attn, v)
                    return mod.output_proj(resampled.transpose(1, 2).flatten(-2))
                return patched

            attn_module.forward = make_patched(attn_module, orig_forward, block_idx)

    def detach(self):
        for block_idx, (mod, orig_forward) in self._orig_forwards.items():
            mod.forward = orig_forward
        self._orig_forwards = {}
        self.records = []


class _ResamplerSteerer:
    """Monkey-patches IdeficsPerceiverAttention.forward on selected blocks to add a
    static additive bias to `scores` before softmax -- the boost_suppress recipe
    reimplemented at the resampler cross-attention site (no attention_mask kwarg
    exists here to intercept via a clean forward-hook the way self_attn allows)."""
    def __init__(self, block_head_bias):
        # block_head_bias: dict[block_idx] -> (head_indices, boost_context_idx, suppress_context_idx)
        self.block_head_bias = block_head_bias
        self._orig_forwards = {}

    def attach(self, resampler):
        for block_idx, (head_indices, boost_idx, suppress_idx) in self.block_head_bias.items():
            attn_module = resampler.blocks[block_idx][0]
            orig_forward = attn_module.forward
            self._orig_forwards[block_idx] = (attn_module, orig_forward)

            def make_patched(mod, boost_idx=boost_idx, suppress_idx=suppress_idx, head_indices=head_indices):
                def patched(context, latents):
                    context_n = mod.context_layer_norm(context)
                    latents_n = mod.latents_layer_norm(latents)
                    batch_size, seq_length, embed_dim = context_n.shape[:3]
                    q = mod.q_proj(latents_n)
                    k = mod.k_proj(torch.cat([context_n, latents_n], dim=-2))
                    v = mod.v_proj(torch.cat([context_n, latents_n], dim=-2))
                    q, k, v = [x.reshape(batch_size, x.shape[1], mod.n_heads, mod.head_dim).transpose(1, 2) for x in (q, k, v)]
                    if mod.qk_layer_norms:
                        q = mod.q_layer_norm(q)
                        k = mod.k_layer_norm(k)
                    scores = torch.einsum("... i d, ... j d -> ... i j", q * mod.qk_scale, k)
                    bias = torch.zeros_like(scores[0, 0])
                    if boost_idx:
                        bias[:, boost_idx] += IDEFICS_SWAP_BIAS
                    if suppress_idx:
                        bias[:, suppress_idx] -= IDEFICS_SWAP_BIAS
                    for h in head_indices:
                        scores[:, h] = scores[:, h] + bias
                    stabilized = scores - scores.amax(dim=-1, keepdim=True).detach()
                    attn = stabilized.softmax(dim=-1)
                    resampled = torch.einsum("... i j, ... j d -> ... i d", attn, v)
                    return mod.output_proj(resampled.transpose(1, 2).flatten(-2))
                return patched

            attn_module.forward = make_patched(attn_module)

    def detach(self):
        for block_idx, (mod, orig_forward) in self._orig_forwards.items():
            mod.forward = orig_forward
        self._orig_forwards = {}


def run_idefics_checkpoint(tag, cfg, n_discovery, n_causal):
    model_id_str = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id_str} ===")
    processor = _AutoProcessor2.from_pretrained(cfg["model_id"])
    model = IdeficsForVisionText2Text.from_pretrained(cfg["model_id"], torch_dtype=torch.bfloat16).to(DEVICE)
    model.eval()
    resampler = model.model.perceiver_resampler
    print(f"{IDEFICS_N_RESAMPLER_BLOCKS} resampler blocks x {IDEFICS_N_RESAMPLER_HEADS} heads")
    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    rng = np.random.RandomState(SEED + 100)
    raw_sum = np.zeros((IDEFICS_N_RESAMPLER_BLOCKS, IDEFICS_N_RESAMPLER_HEADS), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = idefics_prepare_inputs(processor, grid.grid, prompt, DEVICE)
            region_ids, _ = idefics_assign_grid(ROWS, COLS)
            positions = region_positions_from_ids(img_start=0, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = np.array(positions[target_cell], dtype=np.int64)

            recorder = _ResamplerAttentionRecorder()
            recorder.attach(resampler)
            try:
                with torch.no_grad():
                    model(**{k: v for k, v in inputs.items() if k != "pixel_values" or True}, use_cache=False)
            finally:
                recorder.detach()

            for block_idx, attn in recorder.records:
                # attn: (n_heads, n_context) -- context = raw vision patches for this sample's single image
                if attn.shape[-1] > target_positions.max():
                    target_mass = attn[:, target_positions].sum(axis=1)
                    raw_sum[block_idx] += target_mass
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")
    vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)
    ranked = rank_heads_by_score(vis_head_scores)   # here "layer" = resampler block index
    top_blocks_heads = [(r["layer"], r["head"]) for r in ranked[:TOP_K_HEADS]]
    block_head_map = {}
    for b, h in top_blocks_heads:
        block_head_map.setdefault(b, []).append(h)

    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_one(sample, steer):
        inputs = idefics_prepare_inputs(processor, sample["grid"].grid, sample["prompt"], DEVICE)
        prompt_length = int(inputs["input_ids"].shape[1])
        region_ids, _ = idefics_assign_grid(ROWS, COLS)
        positions = region_positions_from_ids(img_start=0, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = list(positions[sample["target_cell"]])
        other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]

        steerer = None
        if steer:
            block_head_bias = {b: (hh, target_positions, other_positions) for b, hh in block_head_map.items()}
            steerer = _ResamplerSteerer(block_head_bias)
            steerer.attach(resampler)
        try:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=IDEFICS_MAX_NEW_TOKENS, do_sample=False,
                                      output_scores=True, return_dict_in_generate=True)
        finally:
            if steerer is not None:
                steerer.detach()

        gen_ids = out.sequences[0, prompt_length:].tolist()
        answer_step = None
        for step in range(len(gen_ids) - 1, -1, -1):
            if gen_ids[step] in all_letter_ids_flat:
                answer_step = step
                break
        if answer_step is None:
            predicted, probs = None, np.zeros(N_OPTIONS)
        else:
            predicted = id_to_letter[gen_ids[answer_step]]
            step_logits = out.scores[answer_step][0].float()
            letter_logits = [step_logits[letter_token_ids[l]].max().item() for l in OPTION_LETTERS]
            probs = torch.softmax(torch.tensor(letter_logits), dim=0).numpy()
        return {"predicted": predicted, "probs": probs, "correct": predicted == sample["correct_letter"]}

    baseline_res, steered_res = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ", leave=False):
        baseline_res.append(run_one(sample, steer=False))
        steered_res.append(run_one(sample, steer=True))

    y_true = [s["correct_letter"] for s in mcq_samples]
    b_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in baseline_res]
    s_pred = [r["predicted"] if r["predicted"] else "UNPARSED" for r in steered_res]
    b_probs = np.stack([r["probs"] for r in baseline_res])
    s_probs = np.stack([r["probs"] for r in steered_res])

    stats = score_stats(vis_head_scores)
    norm_stats = score_stats(normalized_scores(vis_head_scores))
    summary = {
        "family": "idefics", "checkpoint": tag, "model_id": cfg["model_id"],
        "vh_mean": stats["mean"], "vh_median": stats["median"], "vh_std": stats["std"], "vh_max": stats["max"],
        "vh_norm_mean": norm_stats["mean"], "vh_norm_median": norm_stats["median"], "vh_norm_std": norm_stats["std"],
        "baseline_acc": np.mean([r["correct"] for r in baseline_res]),
        "steered_acc": np.mean([r["correct"] for r in steered_res]),
        "baseline_f1": macro_f1(y_true, b_pred), "steered_f1": macro_f1(y_true, s_pred),
        "baseline_auc": macro_auc(y_true, b_probs, OPTION_LETTERS), "steered_auc": macro_auc(y_true, s_probs, OPTION_LETTERS),
    }
    b_acc_v, s_acc_v, s_auc_v = summary["baseline_acc"], summary["steered_acc"], summary["steered_auc"]
    print(f"  [{tag}] baseline_acc={b_acc_v:.3f}  steered_acc={s_acc_v:.3f}  steered_auc={s_auc_v:.3f}")

    free_gpu(model, processor)
    return summary

## Section 4 -- Pilot run

In [ ]:
for tag, cfg in IDEFICS_CHECKPOINTS.items():
    result = run_idefics_checkpoint(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[("idefics", tag + "_PILOT")] = result

pd.DataFrame([v for k, v in all_results.items() if k[0] == "idefics" and "PILOT" in k[1]])

## Section 4 -- Main run

In [ ]:
for tag, cfg in IDEFICS_CHECKPOINTS.items():
    result = run_idefics_checkpoint(tag, cfg, IDEFICS_N_DISCOVERY, IDEFICS_N_CAUSAL)
    all_results[("idefics", tag)] = result

idefics_df = pd.DataFrame([v for k, v in all_results.items() if k[0] == "idefics" and "PILOT" not in k[1]])
idefics_df.to_csv("logs/multistage_idefics_results.csv", index=False)
print(idefics_df.to_string(index=False))

---

# Final summary: all families, all stages

In [ ]:
final_df = pd.DataFrame([v for k, v in all_results.items() if "PILOT" not in k[1]])
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 20)
print(final_df.to_string(index=False))
final_df.to_csv("logs/multistage_vis_head_causality_summary.csv", index=False)
print("\nSaved to logs/multistage_vis_head_causality_summary.csv")